In [2]:
from Functions_wrapped import *
import matplotlib.pyplot as plt



In [2]:
n_compounds=10
differentiate=1
checks_hierarchical=65

In [3]:
#calculate_metrics_hierarchical_fast(n_compounds, differentiate)

In [9]:
hm=calculate_full_metrics_hierarchical_fast(n_compounds, differentiate)

In [10]:
hm[6]

2.0

In [12]:
hm[7]

4.3974

In [9]:
id_samps=np.arange(10)
id_positives=[2]
ratios=[3]

In [10]:
full_iterative_uneven_splitter(id_samps,id_positives,ratios)

(7, 2, 5)

In [5]:
WA_mat = assign_wells_mat(n_compounds=16)

In [6]:
WA_mat

array([[ True, False, False, False,  True, False, False, False],
       [ True, False, False, False, False,  True, False, False],
       [ True, False, False, False, False, False,  True, False],
       [ True, False, False, False, False, False, False,  True],
       [False,  True, False, False,  True, False, False, False],
       [False,  True, False, False, False,  True, False, False],
       [False,  True, False, False, False, False,  True, False],
       [False,  True, False, False, False, False, False,  True],
       [False, False,  True, False,  True, False, False, False],
       [False, False,  True, False, False,  True, False, False],
       [False, False,  True, False, False, False,  True, False],
       [False, False,  True, False, False, False, False,  True],
       [False, False, False,  True,  True, False, False, False],
       [False, False, False,  True, False,  True, False, False],
       [False, False, False,  True, False, False,  True, False],
       [False, False, Fal

In [7]:
full_mean_metrics_fast(well_assigner=WA_mat, differentiate=1)

(8, 0, 1.0, 0.0, 2.0, 6.0)

In [8]:
mean_metrics_fast(well_assigner=WA_mat, differentiate=1)

(8, 0, 1.0, 0.0)

In [ ]:
25/500

In [ ]:
14/700

In [ ]:
10/1000

In [ ]:
7/1400

In [ ]:
5/2500

In [ ]:
3/3000

In [ ]:
N=np.arange(1e5)
rho=0.99

In [ ]:
def f(N,rho):
    MT=((1-rho)**N+(N+1)*(1-(1-rho)**N))/N
    return MT
def f2(N,rho):
    MT=((N+1)-N*(1-rho)**N)/N
    return MT
def f3(N,rho):
    MT=((1/N+1)-(1-rho)**N)
    return MT

In [ ]:
plt.scatter(f2(N,1e-4),f(N,1e-4))

In [ ]:
plt.scatter(f2(N,1e-4),f3(N,1e-4))

In [ ]:
plt.plot(N,f(N,1e-4))

In [ ]:
np.argmin(f(N,1e-4))

In [ ]:

import time

# ── User-defined parameters ──────────────────────────────────────────────────
N_list    = [100, 200, 500, 1000]   # sample sizes to sweep
diff_list = [4,   3,  2,   1]   # differentiation level for each N (same length as N_list)

checks_hierarchical = 500        # MC checks for hierarchical simulation
metrics_checks      = 1e0        # max_checks passed to mean_metrics_fast (internal scaler=1e3)

# Random design parameters (from rand_WA_wrapped.py defaults)
rand_guesses          = 2    # number of random WA candidates to evaluate
rand_max_redundancy   = 2.0   # upper bound on well-redundancy relative to N*log2(N)
rand_min_redundancy   = 0.5   # lower bound on well-redundancy
rand_n_compounds_per_well = 0 # 0 = auto-select
rand_n_wells          = 0     # 0 = auto-select
rand_max_compounds    = 0     # 0 = auto-select (uses get_max_C default)
# ────────────────────────────────────────────────────────────────────────────

rows = []

for N, diff in zip(N_list, diff_list):
    # Maximum number of multidim dimensions that make sense for this N
    # (require L1 = ceil(N^(1/k)) >= 2  =>  k <= log2(N))
    max_dims = max(3, int(np.floor(np.log2(max(N, 2)))))

    # ── Build list of (method_name, assign_fn, kwargs) for deterministic methods
    methods_spec = []

    # Matrix (multidim-2 alias)
    methods_spec.append(('Matrix', assign_wells_mat, {'n_compounds': N}))

    # Multidim 3 … max_dims
    for nd in range(3, max_dims + 1):
        methods_spec.append((f'multidim-{nd}', assign_wells_multidim,
                             {'n_compounds': N, 'n_dims': nd}))

    # Binary
    methods_spec.append(('Binary', assign_wells_bin,
                         {'n_compounds': N, 'differentiate': diff}))

    # Chinese remainder (standard)
    methods_spec.append(('Chinese remainder', assign_wells_chinese,
                         {'n_compounds': N, 'differentiate': diff}))

    # Chinese remainder with backtrack
    methods_spec.append(('Ch. rm. bktrk', assign_wells_chinese,
                         {'n_compounds': N, 'differentiate': diff, 'backtrack': True}))

    # Chinese special (only defined for diff 2 or 3)
    if diff in [2, 3]:
        methods_spec.append(('Chinese special', assign_wells_chinese,
                             {'n_compounds': N, 'differentiate': diff, 'special_diff': True}))

    # STD
    methods_spec.append(('STD', assign_wells_STD,
                         {'n_compounds': N, 'differentiate': diff}))

    # ── Deterministic methods: time WA construction + metrics calculation ─────
    for method_name, wa_fn, wa_kwargs in methods_spec:
        t0 = time.perf_counter()
        WA = wa_fn(**wa_kwargs)
        result = mean_metrics_fast(well_assigner=WA, differentiate=diff,
                                   max_checks=metrics_checks)
        elapsed = time.perf_counter() - t0
        rows.append({'N': N, 'diff': diff, 'Method': method_name,
                     'time': elapsed, 'mean_tests': result[0]})

    # ── Random: assign_wells_random_precomp already returns mean_tests ────────
    t0 = time.perf_counter()
    _, rand_mean_tests, _ = assign_wells_random_precomp(
        n_compounds=N,
        differentiate=diff,
        guesses=rand_guesses,
        max_redundancy=rand_max_redundancy,
        min_redundancy=rand_min_redundancy,
        n_compounds_per_well=rand_n_compounds_per_well,
        n_wells=rand_n_wells,
        max_compounds=rand_max_compounds,
        return_me=True,
    )
    elapsed = time.perf_counter() - t0
    rows.append({'N': N, 'diff': diff, 'Method': 'Random',
                 'time': elapsed, 'mean_tests': rand_mean_tests})

    # ── Hierarchical: calculate_full_metrics_hierarchical_fast ────────────────
    t0 = time.perf_counter()
    hier = calculate_full_metrics_hierarchical_fast(N, diff, checks=checks_hierarchical)
    elapsed = time.perf_counter() - t0
    # hier[0] = mean total experiments
    rows.append({'N': N, 'diff': diff, 'Method': 'Hierarchical',
                 'time': elapsed, 'mean_tests': hier[0]})

timing_df = pd.DataFrame(rows, columns=['N', 'diff', 'Method', 'time', 'mean_tests'])
timing_df['ET'] = timing_df['mean_tests'] / timing_df['N']
timing_df




----------------------------------------------------------------------------------------------------------
Evaluated 326 different random designs each with 2 configurations 




----------------------------------------------------------------------------------------------------------
Evaluated 1092 different random designs each with 2 configurations 


